# Exploration de l'API SNCF

Objectifs et ordre détaillés dans `notebooks/README.md`.
Le point le plus critique est le point 4 : sans identifiant de circulation persistant d'une gare à l'autre, l'hypothèse de propagation ne tient pas.

In [114]:
%reload_ext autoreload
%autoreload 2
from dotenv import load_dotenv

load_dotenv()

from collector.sncf_client import SncfClient
from collector.stations import STATIONS

client = SncfClient()

## 1. Résoudre les id réels (stop_area) des cinq gares via l'endpoint `places`

In [115]:
for station in STATIONS:
    stop_station = client.search_places(station.label)
    try :
        observed_id = stop_station['places'][0]['stop_area']['id']

        if  observed_id != station.id:
            print(f"{station.label}: {station.id} devrait plutôt etre cet id : {observed_id}")
        else:
            print(f"ID correct pour {station.label}: {station.id}")
    except (KeyError, IndexError):
        print(f"Mauvais parsing pour ce json:\n{stop_station}")

Code stop_area correct pour Bordeaux Saint-Jean: stop_area:SNCF:87581009
Code stop_area correct pour Toulouse Matabiau: stop_area:SNCF:87611004
Code stop_area correct pour Montpellier Saint-Roch: stop_area:SNCF:87773002
Code stop_area correct pour Marseille Saint-Charles: stop_area:SNCF:87751008
Code stop_area correct pour Antibes: stop_area:SNCF:87757674


## 2. Appeler `departures` sur une gare et inspecter la structure complète

## 3. Vérifier que `data_freshness=realtime` renvoie un horaire différent du théorique quand un train est en retard

## 4. Identifier le champ portant l'identifiant de circulation persistant d'une gare à l'autre

Point critique : condition nécessaire à tout le projet.

In [116]:
from collector.sncf_client import get_train_id_from_departure, get_train_destination_from_departure, \
    extract_id_from_places_response
%reload_ext autoreload
%autoreload 2
gare_bordeaux = STATIONS[0]
some_departure = client.departures(gare_bordeaux.id)['departures'][0]

train_id = get_train_id_from_departure(some_departure)

# destination random
destination = get_train_destination_from_departure(some_departure)
destination_search_response = client.search_places(destination)
destination_id = extract_id_from_places_response(destination_search_response)

if destination_id:
    arrivals = client.arrivals(destination_id)['arrivals']
    searched_arrival = [arrival for arrival in arrivals if arrival['display_informations']['headsign'] == train_id]
    if len(searched_arrival) > 0 :
        searched_arrival = searched_arrival[0]

        # On vérifie que le point de départ est bien notre gare d'origine :
        links = searched_arrival['display_informations']['links']

        origin_station = [origin for origin in links if origin['rel'] == 'origins'][0]

        if origin_station['id'] == gare_bordeaux.id:
            print("C'est gagné on a bien un train qui a le même id et même origine dans la gare d'arrivé"
                  f"\ntrain_id: {train_id}, destination: {destination_id}, depart: {gare_bordeaux.id}")
        else:
            print("Perdu on dirait que les train ne concordent pas")

    else:
        print("On ne trouve pas de train en provenance de notre gare à l'arrivée: Bizarre")

else :
    print(f"Destination non trouvée parmi les gares")


C'est gagné on a bien un train qui a le même id et même origine dans la gare d'arrivé
train_id: 865130, destination: stop_area:SNCF:87491209, depart: stop_area:SNCF:87581009


## 5. Mesurer le quota réel : combien d'appels avant un HTTP 429

## 6. Vérifier si le quai (`stop_point`) est renseigné en temps réel